# EfficientNet-B3 Reproducibility Notebook

This notebook is designed for GitHub users to run, tune, and test the model themselves.

Default workflow:
1. Set your dataset path.
2. Edit `active_config` or one of the provided configs.
3. Train from the selected config.
4. Evaluate the model trained in this notebook session.

Checkpoint loading is optional. You do **not** need a pre-existing `.pth` file to use this notebook.

## 1. Imports


In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from torchvision.models import efficientnet_b3


## 2. Reproducibility, Device, and Paths


In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

project_root = Path.cwd().parent if Path.cwd().name.lower() in {"notebooks", "notebook"} else Path.cwd()
results_dir = project_root / "results"
checkpoints_dir = project_root / "checkpoints"
plots_dir = results_dir / "plots"

results_dir.mkdir(exist_ok=True)
checkpoints_dir.mkdir(exist_ok=True)
plots_dir.mkdir(exist_ok=True)

# GitHub-friendly dataset setup
# Option 1: set an environment variable before opening the notebook:
#   export DATA_ROOT=/path/to/real-vs-fake
# Option 2: edit DATA_ROOT directly below.
DATA_ROOT = os.environ.get("DATA_ROOT", "")

if DATA_ROOT:
    data_root = Path(DATA_ROOT).expanduser()
else:
    # Common local fallback used during the original experiments.
    data_root = Path.home() / "Desktop" / "real-vs-fake" / "real_vs_fake" / "real-vs-fake"

train_dir = data_root / "train"
val_dir = data_root / "valid"
test_dir = data_root / "test"

print("Project root:", project_root)
print("Data root:", data_root)
print("Train dir exists:", train_dir.exists())
print("Val dir exists:", val_dir.exists())
print("Test dir exists:", test_dir.exists())

if not train_dir.exists() or not val_dir.exists():
    print()
    print("Update DATA_ROOT before training. Expected ImageFolder layout:")
    print("DATA_ROOT/train/<class_name>/image.jpg")
    print("DATA_ROOT/valid/<class_name>/image.jpg")
    print("DATA_ROOT/test/<class_name>/image.jpg")

## 3. Transforms and Dataloaders


In [ ]:
input_size = 224

val_transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Alias used for all evaluation splits.
eval_transform = val_transform

light_transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

medium_transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

strong_transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    transforms.RandomRotation(15),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

TRANSFORMS = {
    "light": light_transform,
    "medium": medium_transform,
    "strong": strong_transform,
}


def validate_imagefolder_split(split_dir, split_name):
    split_dir = Path(split_dir)
    if not split_dir.exists():
        raise FileNotFoundError(f"{split_name} directory does not exist: {split_dir}")
    class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    if not class_dirs:
        raise FileNotFoundError(
            f"{split_name} directory has no class subfolders: {split_dir}. "
            "Expected ImageFolder layout such as fake/ and real/."
        )


def get_dataloaders(augmentation_name="light", batch_size=8, num_workers=2):
    if augmentation_name not in TRANSFORMS:
        raise ValueError(f"Unsupported augmentation setting: {augmentation_name}")

    validate_imagefolder_split(train_dir, "Train")
    validate_imagefolder_split(val_dir, "Validation")

    train_dataset = datasets.ImageFolder(train_dir, transform=TRANSFORMS[augmentation_name])
    val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return train_loader, val_loader

# Quick dataset check. This does not stop the notebook if data is not downloaded yet.
try:
    train_loader, val_loader = get_dataloaders("light", batch_size=8)
    print("Classes:", train_loader.dataset.classes)
    print("Class to index:", train_loader.dataset.class_to_idx)
    print("Train size:", len(train_loader.dataset))
    print("Val size:", len(val_loader.dataset))
except FileNotFoundError as exc:
    print("Dataset folders not found yet. Update DATA_ROOT before training.")
    print(exc)

## 4. Model Definition and Metrics


In [ ]:
def get_efficientnet_b3(num_classes=2):
    model = efficientnet_b3(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model



def compute_metrics(y_true, y_pred, y_prob):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        metrics["roc_auc"] = float("nan")
    return metrics


## 5. Training and Validation Functions


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * images.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = running_loss / len(loader.dataset)
    return metrics


## 6. Experiment Runner


In [ ]:
def run_experiment(config):
    print(f"Starting run: {config['run_id']}")

    run_batch_size = int(config.get("batch_size", 8))
    run_augmentation = config.get("augmentation", "light")

    train_loader, val_loader = get_dataloaders(
        augmentation_name=run_augmentation,
        batch_size=run_batch_size,
    )

    assert train_loader.batch_size == run_batch_size
    assert val_loader.batch_size == run_batch_size

    print("Using augmentation:", run_augmentation)
    print("Using batch size:", run_batch_size)

    model = get_efficientnet_b3(num_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()

    if config["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )
    elif config["optimizer"] == "AdamW":
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )
    elif config["optimizer"] == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(), lr=config["lr"], momentum=0.9, weight_decay=config["weight_decay"]
        )
    else:
        raise ValueError(f"Unsupported optimizer: {config['optimizer']}")

    scheduler = None
    if config.get("scheduler", "none") == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=1,
        )

    best_val_f1 = -1.0
    best_epoch = -1
    patience_counter = 0
    history = []
    checkpoint_path = checkpoints_dir / f"{config['run_id']}_best.pth"
    start_time = time.time()

    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = validate_one_epoch(model, val_loader, criterion, device)

        history_row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
        }
        history.append(history_row)

        print(
            f"Epoch {epoch + 1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

        if scheduler is not None:
            scheduler.step(val_metrics["loss"])

        if np.isnan(train_loss) or np.isnan(val_metrics["loss"]):
            print("Stopping early: loss became NaN")
            break

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "epoch": best_epoch,
                    "best_val_f1": best_val_f1,
                    "class_to_idx": train_loader.dataset.class_to_idx,
                },
                checkpoint_path,
            )
        else:
            patience_counter += 1

        if patience_counter >= config["patience"]:
            print("Early stopping triggered")
            break

    elapsed_minutes = (time.time() - start_time) / 60
    history_df = pd.DataFrame(history)
    history_path = results_dir / f"{config['run_id']}_history.csv"
    history_df.to_csv(history_path, index=False)

    if checkpoint_path.exists():
        best_checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(best_checkpoint["model_state_dict"])
        model.eval()
        print("Loaded best checkpoint into memory for evaluation:", checkpoint_path)

    run_result = {
        "run_id": config["run_id"],
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "checkpoint": str(checkpoint_path),
        "history_csv": str(history_path),
        "elapsed_minutes": elapsed_minutes,
        **config,
    }

    print("Best validation F1:", best_val_f1)
    print("Best epoch:", best_epoch)
    print("Saved checkpoint:", checkpoint_path)
    print("Saved history:", history_path)
    return history_df, run_result, model

## 7. Reproducible Configurations


In [ ]:
baseline_config = {
    "run_id": "E01",
    "run_type": "baseline",
    "epochs": 10,
    "patience": 3,
    "optimizer": "Adam",
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "augmentation": "light",
    "batch_size": 8,
    "scheduler": "none",
    "notes": "baseline EfficientNet-B3 training run",
}

final_config = {
    "run_id": "E07",
    "run_type": "tuning",
    "epochs": 10,
    "patience": 3,
    "optimizer": "AdamW",
    "lr": 3e-4,
    "weight_decay": 1e-3,
    "augmentation": "light",
    "batch_size": 8,
    "scheduler": "none",
    "notes": "final selected EfficientNet-B3 configuration from updated experiments",
}

config_table = pd.DataFrame([baseline_config, final_config])
config_table


### Choose and Run a Configuration

Edit `active_config` below to tune the model. You can start from `baseline_config`, `final_config`, or your own custom dictionary.

Set `RUN_TRAINING = True` when your dataset path is configured. The notebook will train from the config and save a new checkpoint automatically.

In [ ]:
# Main switch for GitHub users.
# Set this to True after DATA_ROOT points to your dataset.
RUN_TRAINING = False

# Start from the selected final config, then edit any values you want to tune.
active_config = final_config.copy()
active_config.update({
    # Examples:
    # "run_id": "my_custom_run",
    # "epochs": 5,
    # "lr": 1e-4,
    # "augmentation": "medium",
    # "batch_size": 16,
})

model = None
checkpoint = None
history_df = None
run_result = None

if RUN_TRAINING:
    history_df, run_result, model = run_experiment(active_config)
    run_result
else:
    print("Training is disabled. Set RUN_TRAINING = True after configuring DATA_ROOT and active_config.")

## 8. Plot Helpers


In [ ]:
def plot_history(history_df, run_id, save=True):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{run_id} Loss Curves")
    plt.legend()
    plt.grid(True)
    if save:
        plt.savefig(plots_dir / f"{run_id}_loss_curves.png", bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["val_f1"], marker="o", label="Val F1")
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")
    plt.title(f"{run_id} Validation F1")
    plt.legend()
    plt.grid(True)
    if save:
        plt.savefig(plots_dir / f"{run_id}_val_f1.png", bbox_inches="tight")
    plt.show()


def plot_confusion_matrix(model, loader, class_names, title, save_name=None):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    plt.figure(figsize=(6, 6))
    disp.plot(cmap="Blues", values_format="d")
    plt.title(title)
    if save_name is not None:
        plt.savefig(plots_dir / save_name, bbox_inches="tight")
    plt.show()


## 9. Evaluation Helpers and Optional Checkpoint Loading

The notebook does not require a checkpoint to start. If you run training above, evaluation uses the trained model from memory.

Only set `LOAD_EXISTING_CHECKPOINT = True` if you already have a `.pth` file in `checkpoints/` or a path to one.

In [ ]:
def build_eval_loader(dataset_root, batch_size=8, num_workers=2):
    dataset_root = Path(dataset_root)
    validate_imagefolder_split(dataset_root, "Evaluation")
    dataset = datasets.ImageFolder(dataset_root, transform=eval_transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    print("Dataset:", dataset_root)
    print("Classes:", dataset.classes)
    print("Class to index:", dataset.class_to_idx)
    print("Total images:", len(dataset))
    return dataset, loader


def load_checkpoint_if_available(model_fn, checkpoint_name_or_path):
    if checkpoint_name_or_path is None:
        print("No checkpoint requested.")
        return None, None

    checkpoint_path = Path(checkpoint_name_or_path).expanduser()
    if not checkpoint_path.is_absolute():
        checkpoint_path = checkpoints_dir / checkpoint_path

    if not checkpoint_path.exists():
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Skipping checkpoint loading. Train a model first or provide an existing checkpoint path.")
        return None, None

    checkpoint = torch.load(checkpoint_path, map_location=device)
    loaded_model = model_fn(num_classes=2)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    print("Loaded checkpoint:", checkpoint_path)
    print("Best validation epoch:", checkpoint.get("epoch"))
    print("Best validation F1:", checkpoint.get("best_val_f1"))
    return loaded_model, checkpoint


def evaluate_loader(model, loader):
    criterion = nn.CrossEntropyLoss()
    metrics = validate_one_epoch(model, loader, criterion, device)
    return pd.DataFrame([metrics])


LOAD_EXISTING_CHECKPOINT = False
CHECKPOINT_NAME_OR_PATH = "E07_best.pth"

if LOAD_EXISTING_CHECKPOINT:
    loaded_model, loaded_checkpoint = load_checkpoint_if_available(get_efficientnet_b3, CHECKPOINT_NAME_OR_PATH)
    if loaded_model is not None:
        model = loaded_model
        checkpoint = loaded_checkpoint
elif model is None:
    print("No checkpoint loaded. To evaluate, either run training above or set LOAD_EXISTING_CHECKPOINT = True.")
else:
    print("Using the model trained in this notebook session for evaluation.")

## 10. Held-Out Test Split Evaluation

This section evaluates the current `model`. Run training first, or optionally load a checkpoint in the previous section.

In [ ]:
if model is None:
    print("No model is available. Run training first or load an existing checkpoint.")
elif not test_dir.exists():
    print("Test directory not found. Update DATA_ROOT or test_dir before evaluation:", test_dir)
else:
    test_dataset, test_loader = build_eval_loader(test_dir, batch_size=active_config["batch_size"])
    test_results = evaluate_loader(model, test_loader)
    display(test_results.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
        "loss": "{:.6f}",
    }))

In [ ]:
if model is None:
    print("No model is available. Run training first or load an existing checkpoint.")
elif "test_loader" not in globals() or "test_dataset" not in globals():
    print("Run the test evaluation cell first to create test_loader and test_dataset.")
else:
    plot_confusion_matrix(
        model,
        test_loader,
        test_dataset.classes,
        "EfficientNet-B3 Test Confusion Matrix",
        "efficientnet_b3_test_confusion_matrix.png",
    )

## 11. Optional: Unseen Dataset A Evaluation

Set `RUN_UNSEEN_A = True` and update `unseen_a_root` if you want to evaluate an additional ImageFolder-style dataset.

In [ ]:
RUN_UNSEEN_A = False
unseen_a_root = Path.home() / "Desktop" / "130K" / "images"

if not RUN_UNSEEN_A:
    print("Skipping optional unseen a evaluation. Set RUN_UNSEEN_A = True to run it.")
elif model is None:
    print("No model is available. Run training first or load an existing checkpoint.")
elif not Path(unseen_a_root).exists():
    print("Optional dataset directory not found:", unseen_a_root)
else:
    unseen_a_dataset, unseen_a_loader = build_eval_loader(unseen_a_root, batch_size=active_config["batch_size"])
    unseen_a_results = evaluate_loader(model, unseen_a_loader)
    display(unseen_a_results)

In [ ]:
if not RUN_UNSEEN_A:
    print("Skipping optional confusion matrix. Set RUN_UNSEEN_A = True and run the evaluation cell first.")
elif model is None:
    print("No model is available. Run training first or load an existing checkpoint.")
elif "unseen_a_loader" not in globals() or "unseen_a_dataset" not in globals():
    print("Run the optional evaluation cell first.")
else:
    plot_confusion_matrix(
        model,
        unseen_a_loader,
        unseen_a_dataset.classes,
        "EfficientNet-B3 Unseen Dataset A Confusion Matrix",
        "efficientnet_b3_unseen_a_confusion_matrix.png",
    )

## 12. Optional: Unseen Dataset B Preparation


In [ ]:
# Optional helper for datasets distributed as Parquet rows with image bytes.
# You only need this if your Unseen Dataset B is not already arranged as:
# Deepfake_vs_Real_eval/fake/... and Deepfake_vs_Real_eval/real/...

# Example usage:
# export_parquet_images_to_imagefolder(
#     parquet_path=Path.home() / "Desktop" / "Deepfake_vs_Real.parquet",
#     output_root=Path.home() / "Desktop" / "Deepfake_vs_Real_eval",
#     image_column="image",
#     label_column="label",
# )

def export_parquet_images_to_imagefolder(parquet_path, output_root, image_column="image", label_column="label", max_rows=None):
    from io import BytesIO
    from PIL import Image

    parquet_path = Path(parquet_path)
    output_root = Path(output_root)
    df = pd.read_parquet(parquet_path)
    if max_rows is not None:
        df = df.head(max_rows)

    output_root.mkdir(parents=True, exist_ok=True)
    for idx, row in df.iterrows():
        label = str(row[label_column]).lower()
        class_name = "fake" if label in {"fake", "1", "deepfake"} else "real"
        class_dir = output_root / class_name
        class_dir.mkdir(parents=True, exist_ok=True)

        image_value = row[image_column]
        if isinstance(image_value, dict) and "bytes" in image_value:
            image_bytes = image_value["bytes"]
        else:
            image_bytes = image_value

        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        image.save(class_dir / f"{idx}.jpg")

    print("Exported", len(df), "images to", output_root)


## 13. Optional: Unseen Dataset B Evaluation

Set `RUN_UNSEEN_B = True` and update `unseen_b_root` if you want to evaluate this additional dataset.

In [ ]:
RUN_UNSEEN_B = False
unseen_b_root = Path.home() / "Desktop" / "Deepfake_vs_Real_eval"

if not RUN_UNSEEN_B:
    print("Skipping optional unseen b evaluation. Set RUN_UNSEEN_B = True to run it.")
elif model is None:
    print("No model is available. Run training first or load an existing checkpoint.")
elif not Path(unseen_b_root).exists():
    print("Optional dataset directory not found:", unseen_b_root)
else:
    unseen_b_dataset, unseen_b_loader = build_eval_loader(unseen_b_root, batch_size=active_config["batch_size"])
    unseen_b_results = evaluate_loader(model, unseen_b_loader)
    display(unseen_b_results)

In [ ]:
if not RUN_UNSEEN_B:
    print("Skipping optional confusion matrix. Set RUN_UNSEEN_B = True and run the evaluation cell first.")
elif model is None:
    print("No model is available. Run training first or load an existing checkpoint.")
elif "unseen_b_loader" not in globals() or "unseen_b_dataset" not in globals():
    print("Run the optional evaluation cell first.")
else:
    plot_confusion_matrix(
        model,
        unseen_b_loader,
        unseen_b_dataset.classes,
        "EfficientNet-B3 Unseen Dataset B Confusion Matrix",
        "efficientnet_b3_unseen_b_confusion_matrix.png",
    )

## 14. Notes for GitHub Users

- No pre-existing checkpoint is required. The default path is to edit `active_config`, train, and evaluate the model from the notebook session.
- If you want to use an existing checkpoint, place it in `checkpoints/` and set `LOAD_EXISTING_CHECKPOINT = True`.
- The original experiments used image-level binary classification with `ImageFolder`, so each dataset split should contain class subfolders such as `fake/` and `real/`.
- You can set the dataset path with the `DATA_ROOT` environment variable or by editing the `DATA_ROOT` cell near the top of the notebook.
- The held-out test split should only be used after model selection.
- External unseen datasets can have strong domain shift. Treat those results as generalization checks, not as tuning feedback.
- To make exact runs more deterministic, keep the same PyTorch, torchvision, CUDA, and GPU environment where possible.